In [13]:
import math
from collections import Counter, defaultdict

# This is a blueprint for the TFIDFVectorizer object.
class TFIDFVectorizer:
    # Constructor method.
    def __init__(self, v_max=None, lowercase=True):
        """
        Initializes the TF-IDF Vectorizer.

        Args:
            v_max (int, optional): Maximum size of the vocabulary. Keeps the most
                                   frequent v_max terms.

                                   Suppose you have millions of documents and hundreds
                                   of thousands of unique words.

                                   Your TF-IDF vectors could be huge and sparse (mostly
                                   zeros), consuming a lot of memory.
                                   
                                   Limiting v_max keeps the vector size manageable.
            lowercase (bool): Whether to lowercase documents before tokenization.
        """

        # The following instance variables are unique to each
        # instance of this class.
        self.v_max = v_max

        # self.lowercase is the instance variable that belongs
        # to the instance of the class.

        # lowercase is the parameter that is assigned to the
        # self.lowercase instance variable.
        self.lowercase = lowercase

        # List data structure.
        self.vocabulary = []

        # Dictionary data structures.
        self.idf = {}
        self.word_to_index = {}

    def _tokenize(self, document):
        """Splits a document into tokens."""
        if self.lowercase:
            document = document.lower()

        # print('In _tokenize:\ndocument.split():', document.split(), '\n')

        return document.split()

    def fit(self, corpus: list[str]):
        """
        Learns the vocabulary and computes IDF values.

        Args:
            corpus (list of str): The collection of documents to fit on.
        """
        # Tokenize all documents by iterating over each of the
        # items/elements in the corpus list.
        tokenized_docs = [self._tokenize(doc) for doc in corpus]

        print('tokenized_docs:\n', tokenized_docs)

        # Count word frequencies across entire corpus.
        # term_freq = Counter(word for doc in tokenized_docs for word in doc)

        # Initialize an empty Counter
        term_freq = Counter()
    
        # Loop through each document
        for doc in tokenized_docs:
            # Loop through each word in the document
            for word in doc:
                # Add the word to the Counter
                term_freq[word] += 1

        print('\nterm_freq:\n', term_freq)

        # If vocabulary limit is set, keep top v_max most frequent terms.
        if self.v_max:
            most_common = term_freq.most_common(self.v_max)
            self.vocabulary = [term for term, _ in most_common]
        else:
            # Sort the vocabulary words in alphabetical order.

            # The keys are unique in the term_freq dictionary,
            # but they map to the number of times they appear
            # in all of the documents inside "corpus".
            self.vocabulary = sorted(term_freq.keys())

        print('\nself.vocabulary:\n', self.vocabulary)
        
        # Create word-to-index mapping.
        # self.word_to_index = {word: i for i, word in enumerate(self.vocabulary)}

        # Initialize an empty dictionary.
        self.word_to_index = {}
        
        # Loop over each word in the vocabulary.
        # Create a dictionary where the keys are the words
        # and the values are the index where each word is located
        # in the self.vocabulary list.
        for i in range(len(self.vocabulary)):
            # Get the word at index i
            word = self.vocabulary[i]

            # Map the word to its index.
            self.word_to_index[word] = i

        print('\nself.word_to_index:\n', self.word_to_index)

        # Compute document frequency for each term.

        # Passing int as an argument to defaultdict means that
        # the default value for a key that doesn't exist is 0.
        doc_freq = defaultdict(int)

        for word in self.vocabulary:
            for doc in tokenized_docs:
                # if 'a' in ['the', 'hotel', 'and', 'the', 'stay', 'were', 'great']:
                #   doc_freq['a'] = doc_freq['a'] + 1
                if word in doc:
                    doc_freq[word] += 1

        '''
        self.vocabulary:
        ['a', 'and', 'destination', 'great', 'hotel', 'in',
         'stay', 'the', 'this', 'was', 'were']

        tokenized_docs:
        [
            ['the', 'hotel', 'and', 'the', 'stay', 'were', 'great'],
            ['this', 'was', 'a', 'great', 'stay'],
            ['great', 'stay', 'in', 'a', 'great', 'destination'],
            ['great', 'destination']
        ]
        '''

        '''
        doc_freq:
        defaultdict(<class 'int'>, {
            'a': 2, 'and': 1, 'destination': 2, 'great': 4,
            'hotel': 1, 'in': 1, 'stay': 3, 'the': 1,
            'this': 1, 'was': 1, 'were': 1
        })
        '''
        print('\ndoc_freq:\n', doc_freq)

        # Compute IDF for each term.
        # The IDF or "Inverse Document Frequency" measures how rare
        # or common a word is across the entire collection of documents.

        # N = 4 because there are four items in the "corupus" list.
        N = len(corpus)

        '''
        self.vocabulary:
        ['a', 'and', 'destination', 'great', 'hotel',
         'in', 'stay', 'the', 'this', 'was', 'were']
        '''
        for word in self.vocabulary:
            # self.idf['a'] = math.log(N / (1 + doc_freq['a'])) + 1

            # +1 smoothing
            self.idf[word] = math.log(N / (1 + doc_freq[word])) + 1

    def transform(self, corpus):
        """
        Transforms a list of documents into TF-IDF vectors.

        Args:
            corpus (list of str): Collection of documents to transform.

        Returns:
            list of lists: TF-IDF matrix representation of the corpus.
        """
        vectors = []
        for doc in corpus:
            tokens = self._tokenize(doc)
            tf_counts = Counter(tokens)
            doc_vector = [0.0] * len(self.vocabulary)
            for word, count in tf_counts.items():
                if word in self.word_to_index:
                    idx = self.word_to_index[word]
                    tf = count  # raw term frequency
                    idf = self.idf[word]
                    doc_vector[idx] = tf * idf
            vectors.append(doc_vector)
        return vectors

    def fit_transform(self, corpus):
      """
      Fits the model and transforms the corpus in one step.

      Args:
        corpus (list of str): Collection of documents to fit and transform.

      Returns:
        list of lists: TF-IDF matrix representation of the corpus.
      """
      self.fit(corpus)
      
      return self.transform(corpus)

In [14]:
# Collection of documents.
corpus = [
    'The hotel and the stay were great',
    'This was a great stay',
    'Great stay in a great destination',
    'Great destination'
]

# Create an object/instance of TFIDFVectorizer class.

# lowercase=False is a keyword argument because the name
# of the parameter "lowercase" is explicitly written.
# Whether to lowercase documents before tokenization.
vectorizer = TFIDFVectorizer(lowercase=True)

# Invoke the fit_transform() method on the object
# named "vectorizer".
tfidf_matrix = vectorizer.fit_transform(corpus)

print("Vocabulary:", vectorizer.vocabulary)
for row in tfidf_matrix:
    print(row)

tokenized_docs:
 [['the', 'hotel', 'and', 'the', 'stay', 'were', 'great'], ['this', 'was', 'a', 'great', 'stay'], ['great', 'stay', 'in', 'a', 'great', 'destination'], ['great', 'destination']]

term_freq:
 Counter({'great': 5, 'stay': 3, 'the': 2, 'a': 2, 'destination': 2, 'hotel': 1, 'and': 1, 'were': 1, 'this': 1, 'was': 1, 'in': 1})

self.vocabulary:
 ['a', 'and', 'destination', 'great', 'hotel', 'in', 'stay', 'the', 'this', 'was', 'were']

self.word_to_index:
 {'a': 0, 'and': 1, 'destination': 2, 'great': 3, 'hotel': 4, 'in': 5, 'stay': 6, 'the': 7, 'this': 8, 'was': 9, 'were': 10}

doc_freq:
 defaultdict(<class 'int'>, {'a': 2, 'and': 1, 'destination': 2, 'great': 4, 'hotel': 1, 'in': 1, 'stay': 3, 'the': 1, 'this': 1, 'was': 1, 'were': 1})
Vocabulary: ['a', 'and', 'destination', 'great', 'hotel', 'in', 'stay', 'the', 'this', 'was', 'were']
[0.0, 1.6931471805599454, 0.0, 0.7768564486857903, 1.6931471805599454, 0.0, 1.0, 3.386294361119891, 0.0, 0.0, 1.6931471805599454]
[1.28768207